In [16]:
import pandas as pd, glob, re, os
import matplotlib.pyplot as plt

EXP_DIR = "../nap_verify/experiments"

def classify_rule(name):
    name = str(name)
    if name in ("none (baseline)", "Pure ε-ball (Row 1)"): return "baseline"
    for ut in ["ALWAYS_ON+OFF", "ALWAYS_ON", "ALWAYS_OFF"]:
        if name.startswith(ut): return ut
    m = re.match(r"Impl (L\d→L\d) \[(.+?)\]", name)
    if m: return f"Impl {m.group(1)} [{m.group(2)}]"
    for it in ["!A->!B", "A->!B", "!A->B", "A->B"]:
        if name.startswith(it): return it
    return "other"

records = []
for path in sorted(glob.glob(os.path.join(EXP_DIR, "*/verification_results/table3_results.csv"))):
    exp_name = path.split("/")[-3]
    m = re.search(r'ACASXU_run2a_(\d+)_(\d+)', exp_name)
    model = f"N{m.group(1)},{m.group(2)}" if m else exp_name
    etype = "impl_ablation" if "impl_ablation" in exp_name else \
            (re.search(r'(L\dL\d)', exp_name).group(1) if re.search(r'L\dL\d', exp_name) else "original")
    alpha = re.search(r'alpha([\d.]+)', exp_name)
    alpha = alpha.group(1) if alpha else "?"
    try:
        df = pd.read_csv(path)
        if 'description' in df.columns: df = df.rename(columns={'description': 'table'})
        if 'true_class' in df.columns and 'class_id' not in df.columns: df['class_id'] = df['true_class']
        df['model'] = model; df['exp_type'] = etype; df['alpha_dir'] = alpha
        df['rule'] = df['table'].apply(classify_rule)
        records.append(df[['model','exp_type','alpha_dir','rule','epsilon','class_id','target_label','result']])
    except: pass

DF = pd.concat(records, ignore_index=True)
DF['epsilon'] = pd.to_numeric(DF['epsilon'], errors='coerce').round(3)
DF['class_id'] = pd.to_numeric(DF['class_id'], errors='coerce')
print(f"✓ {len(DF):,} 行 | {DF['model'].nunique()} 模型 | exp_types: {sorted(DF['exp_type'].unique())}")


✓ 435,200 行 | 48 模型 | exp_types: ['L0L1', 'L1L2', 'L2L3', 'L3L4', 'L4L5', 'impl_ablation', 'original']


In [17]:
# 计算每个 (model, exp_type, alpha_dir, epsilon, rule) 的 Y%
def yrate(sub):
    valid = sub[sub['result'] != '-']
    if len(valid) == 0: return float('nan')
    return (valid['result'] == 'Y').sum() / len(valid) * 100

stats = (DF.groupby(['model','exp_type','alpha_dir','epsilon','rule'])
           .apply(yrate, include_groups=False)
           .reset_index(name='Y_pct'))

# 提取 baseline Y%，计算每条规则的 delta = rule_Y% - baseline_Y%
baseline = stats[stats['rule'] == 'baseline'][['model','exp_type','alpha_dir','epsilon','Y_pct']]\
               .rename(columns={'Y_pct': 'baseline_Y'})
merged = stats[stats['rule'] != 'baseline'].merge(baseline, on=['model','exp_type','alpha_dir','epsilon'], how='left')
merged['delta'] = merged['Y_pct'] - merged['baseline_Y']

# ── 1. 各规则平均提升量（最直接的总览）
rule_summary = (merged.groupby('rule')['delta']
                      .agg(['mean','std','count'])
                      .sort_values('mean', ascending=False)
                      .round(2))
rule_summary.columns = ['avg_delta(%)', 'std', 'n_cases']
print("=== 各规则平均 ΔY% (相比 baseline) ===")
print(rule_summary.to_string())


=== 各规则平均 ΔY% (相比 baseline) ===
                     avg_delta(%)    std  n_cases
rule                                             
Impl L0→L1 [!A->!B]         13.70  21.31       80
Impl L0→L1 [A->!B]          12.89  21.29       80
ALWAYS_ON+OFF               11.05  20.27      292
ALWAYS_OFF                   9.59  18.48      292
Impl L1→L2 [!A->!B]          7.83  17.23       80
Impl L1→L2 [A->!B]           7.17  15.63       80
Impl L3→L4 [!A->!B]          6.88  20.37       80
other                        6.14  12.71     2059
Impl L0→L1 [A->B]            5.33  12.14       80
Impl L2→L3 [!A->!B]          5.02  12.87       80
Impl L2→L3 [A->!B]           4.52  11.51       80
Impl L3→L4 [A->!B]           4.19  12.56       80
Impl L0→L1 [!A->B]           3.71   9.81       80
Impl L1→L2 [A->B]            3.38  10.14       80
ALWAYS_ON                    3.26   9.35      292
Impl L1→L2 [!A->B]           3.02   9.69       80
Impl L3→L4 [!A->B]           1.75   8.39       80
Impl L3→L4 [A->B] 

In [18]:
# 每个模型：最优规则带来的最大 delta
best_per_model = (merged[merged['delta'] > 0]
                  .groupby('model')
                  .agg(max_delta=('delta','max'),
                       best_rule=('rule', lambda x: x.iloc[merged.loc[x.index,'delta'].values.argmax()]),
                       n_improved=('delta', lambda x: (x>5).sum()))
                  .sort_values('max_delta', ascending=False)
                  .head(20))
print("=== 受益最大的模型 TOP 20 ===")
print(best_per_model.round(2).to_string())


=== 受益最大的模型 TOP 20 ===
       max_delta            best_rule  n_improved
model                                            
N2,9      100.00  Impl L0→L1 [!A->!B]          22
N2,8      100.00  Impl L3→L4 [!A->!B]          18
N2,1       81.82                other          56
N3,2       80.95                other          64
N1,5       80.00  Impl L0→L1 [!A->!B]          57
N2,5       80.00           ALWAYS_OFF          29
N5,7       76.56                other          12
N2,3       75.00  Impl L1→L2 [!A->!B]          26
N3,1       75.00           ALWAYS_OFF          24
N4,7       75.00                other          16
N4,1       72.73                other          17
N2,4       71.43                other          19
N5,3       66.67                other          18
N3,7       63.64                other          12
N2,7       63.64                other          24
N4,6       62.50                other          18
N2,2       61.90                other          41
N5,1       61.90           